In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

# we keep only the samples with lung cancer (include adjacent lung and cell lines if you need them)
metadata_df = metadata_df[metadata_df['tissue'] == 'lung cancer']
metadata_df

,sample_title,tissue,cell line,genotype/variation,date.of.birth,gender,op.date,t,n,m,Stage,histology,death.event,death.time..month.,relapse.event.1,relapse.time..month.
samples,,,,,,,,,,,,,,,,
GSM3405969,A271,lung cancer,NaN,NaN,9/28/1941,F,3/26/1996,2,2,0,3b,Adenocarcinoma,1,39,1,17.46666667
GSM3405970,A272,lung cancer,NaN,NaN,9/11/1947,F,4/1/1996,NaN,2,NaN,3b,Adenocarcinoma,1,6.533333333,1,4.9
GSM3405971,A306,lung cancer,NaN,NaN,2/20/1948,F,12/12/1996,3,1,0,3a,adenocarcinoma,1,66.7,1,58.16666667
GSM3405972,A438,lung cancer,NaN,NaN,6/16/1938,F,11/2/2000,2,0,1,4,"adenocarcinoma, BAC",1,50.13333333,0,50.13333333
GSM3405973,A442,lung cancer,NaN,NaN,6/16/1945,F,11/23/2000,2,2,0,3a,adenocarcinoma,1,40.96666667,1,33.33333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM3406045,S646,lung cancer,NaN,NaN,5/26/1933,M,2/12/2007,2,0,0,1B,squamous cell carcinoma,1,24.7,1,9.866666667
GSM3406046,S662,lung cancer,NaN,NaN,4/18/1947,M,9/10/2007,2,0,0,1B,squamous cell carcinoma,0,65.56666667,0,65.56666667
GSM3406047,S666,lung cancer,NaN,NaN,1/6/1930,M,5/22/2008,2,0,0,1B,squamous cell carcinoma,1,81.9,0,81.9


# Use NCBI provided expression data

In [2]:
# Raw counts
ncbi_data_url = "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc=GSE120622&format=file&file=GSE120622_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined = metadata_df.join(df_raw_counts, how='inner')

df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_92062/3356432576.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,tissue,cell line,genotype/variation,date.of.birth,gender,op.date,t,n,m,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM3405969,A271,lung cancer,NaN,NaN,9/28/1941,F,3/26/1996,2,2,0,...,196081,28,40,63,51248,19444,403,60150,20,1176
GSM3405970,A272,lung cancer,NaN,NaN,9/11/1947,F,4/1/1996,NaN,2,NaN,...,173728,22,42,61,28259,8953,243,98710,52,1158
GSM3405971,A306,lung cancer,NaN,NaN,2/20/1948,F,12/12/1996,3,1,0,...,176059,11,15,25,20599,7640,189,73905,73,1617
GSM3405972,A438,lung cancer,NaN,NaN,6/16/1938,F,11/2/2000,2,0,1,...,117460,13,25,58,17735,6705,179,36081,20,431
GSM3405973,A442,lung cancer,NaN,NaN,6/16/1945,F,11/23/2000,2,2,0,...,210753,49,95,165,30906,18833,859,70175,161,1356
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM3406045,S646,lung cancer,NaN,NaN,5/26/1933,M,2/12/2007,2,0,0,...,205217,23,26,37,25401,9101,224,66105,121,908
GSM3406046,S662,lung cancer,NaN,NaN,4/18/1947,M,9/10/2007,2,0,0,...,393386,19,34,63,48319,16616,387,129956,66,1702
GSM3406047,S666,lung cancer,NaN,NaN,1/6/1930,M,5/22/2008,2,0,0,...,171781,10,19,28,12106,2471,31,37379,88,790


# Generate description

In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)
    print("Description generated")

desc_data['data_summary'] = {
    'samples': metadata_df.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated


# Use author provided expression data

# TODO